In [ ]:
import torch
print(torch.cuda.is_available())

True


In [ ]:
# !pip install -U "transformers>=4.39.0"
# !pip install peft bitsandbytes
# !pip install -U "trl>=0.8.3"
# !pip install datasets

In [ ]:
# !pip install tensorboard

In [ ]:
############################################## Matching dataset format

In [ ]:
import os
import json
import torch
from transformers import AutoTokenizer, AutoProcessor, TrainingArguments, LlavaForConditionalGeneration, BitsAndBytesConfig
from trl import SFTTrainer, SFTConfig  # Import SFTConfig here
from peft import LoraConfig
from datasets import Dataset, DatasetDict
from PIL import Image

from peft import prepare_model_for_kbit_training, get_peft_model

In [ ]:
import os
import json
from PIL import Image
from io import BytesIO
from datasets import Dataset

# Define paths
image_folder = r"C:\Users\User\Pictures\Thesis\VQA dataset added\combined-img"
train_json_path = r"C:\Users\User\Pictures\Thesis\VQA json files\training-split\train-melanoma-vascular-pox.json"
val_json_path = r"C:\Users\User\Pictures\Thesis\VQA json files\training-split\validation-accuracy.json"
#train_json_path = r"C:\Users\User\Pictures\Thesis\VQA json files\training-split\test.json"

# Function to convert your dataset into the required format
def convert_vqa_to_dialog_format(json_path, image_folder):
    with open(json_path, 'r') as f:
        data = json.load(f)

    dialogues = []

    for item in data:
        image_name = item['image_name']
        image_path = os.path.join(image_folder, image_name)

        # Load image using PIL
        try:
            image = Image.open(image_path)
        except Exception as e:
            print(f"Error opening image {image_name}: {e}")
            continue  # Skip if the image can't be opened

        # Construct the dialogue exchanges
        dialogue = []

        # User asks the question and provides image (add image to user role)
        dialogue.append({
            "role": "user",
            "content": [
                {
                    "index": None,
                    "text": item['question'],
                    "type": "text"
                },
                {
                    "index": 0,  # This is added to match the reference format
                    "text": None,  # No text for image, just indicate it's an image
                    "type": "image"  # Indicate that it's an image content
                }
            ]
        })

        # Assistant answers the question
        dialogue.append({
            "role": "assistant",
            "content": [{
                "index": None,
                "text": item['answer'],
                "type": "text"
            }]
        })

        # Include the image as a PIL image object (this is what you need)
        dialogues.append({
            'messages': dialogue,
            'images': [image]  # Store image as PIL object
        })

    return dialogues

# Convert train and validation data
train_dialogues = convert_vqa_to_dialog_format(train_json_path, image_folder)
val_dialogues = convert_vqa_to_dialog_format(val_json_path, image_folder)

# Create datasets from the converted dialogues
train_dataset = Dataset.from_dict({
    'messages': [dialogue['messages'] for dialogue in train_dialogues],
    'images': [dialogue['images'] for dialogue in train_dialogues]
})

val_dataset = Dataset.from_dict({
    'messages': [dialogue['messages'] for dialogue in val_dialogues],
    'images': [dialogue['images'] for dialogue in val_dialogues]
})

# Display the first item of the train dataset to verify
print(train_dataset[0]['messages'][0])
print(train_dataset[0]['messages'][1])


{'content': [{'index': None, 'text': 'What is the name for this disease?', 'type': 'text'}, {'index': 0, 'text': None, 'type': 'image'}], 'role': 'user'}
{'content': [{'index': None, 'text': 'melanoma', 'type': 'text'}], 'role': 'assistant'}


In [ ]:
print(train_dataset[0])

{'messages': [{'content': [{'index': None, 'text': 'What is the name for this disease?', 'type': 'text'}, {'index': 0, 'text': None, 'type': 'image'}], 'role': 'user'}, {'content': [{'index': None, 'text': 'melanoma', 'type': 'text'}], 'role': 'assistant'}], 'images': [{'bytes': None, 'path': 'C:\\Users\\User\\Pictures\\Thesis\\VQA dataset added\\combined-img\\ISIC_7623407.jpg'}]}


In [ ]:
print(train_dataset[0]['messages'][0])
print(train_dataset[0]['messages'][1])
print(train_dataset[0]['images'])

{'content': [{'index': None, 'text': 'What is the name for this disease?', 'type': 'text'}, {'index': 0, 'text': None, 'type': 'image'}], 'role': 'user'}
{'content': [{'index': None, 'text': 'melanoma', 'type': 'text'}], 'role': 'assistant'}
[{'bytes': None, 'path': 'C:\\Users\\User\\Pictures\\Thesis\\VQA dataset added\\combined-img\\ISIC_7623407.jpg'}]


In [ ]:
print(train_dataset[0]['images'][0]['path'])
img_path= train_dataset[0]['images'][0]['path']
img = Image.open(img_path)
print(img)
#img.show()

C:\Users\User\Pictures\Thesis\VQA dataset added\combined-img\ISIC_7623407.jpg
<PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=512x512 at 0x1A946163EB0>


In [ ]:
aibot_model_id = "llava-hf/llava-1.5-7b-hf"

# quantization_config = BitsAndBytesConfig(
#     load_in_4bit=True,
# )
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16
)


model = LlavaForConditionalGeneration.from_pretrained(aibot_model_id,
                                                      quantization_config=quantization_config,
                                                      torch_dtype=torch.float16)


#LLAVA_CHAT_TEMPLATE = """A chat between a curious user and an artificial intelligence assistant. The assistant gives helpful, detailed, and polite answers to the user's questions. {% for message in messages %}{% if message['role'] == 'user' %}USER: {% else %}ASSISTANT: {% endif %}{% for item in message['content'] %}{% if item['type'] == 'text' %}{{ item['text'] }}{% elif item['type'] == 'image' %}<image>{% endif %}{% endfor %}{% if message['role'] == 'user' %} {% else %}{{eos_token}}{% endif %}{% endfor %}""" #####
LLAVA_CHAT_TEMPLATE = """{% for message in messages %}{% if message['role'] == 'user' %}USER: {% else %}ASSISTANT: {% endif %}{% for item in message['content'] %}{% if item['type'] == 'text' %}{{ item['text'] }}{% elif item['type'] == 'image' %}<image>{% endif %}{% endfor %}{% if message['role'] == 'user' %} {% else %}{{eos_token}}{% endif %}{% endfor %}""" ###

tokenizer = AutoTokenizer.from_pretrained(aibot_model_id)
tokenizer.chat_template = LLAVA_CHAT_TEMPLATE
processor = AutoProcessor.from_pretrained(aibot_model_id)
processor.tokenizer = tokenizer


class LLavaDataCollator:
    def __init__(self, processor):
        self.processor = processor

    def __call__(self, examples):
        texts = []
        images = []
        for example in examples:
            messages = example["messages"]
            text = self.processor.tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=False
            )
            texts.append(text)
            #images.append(example["images"][0])
            images.append(Image.open(example["images"][0]['path']))

        batch = self.processor(images, texts, return_tensors="pt", padding=True)

        #batch = self.processor(images=images, text=texts, padding=True, truncation=True, max_length=32, return_tensors="pt") #256##
        #print(batch)
        #batch_text = self.processor.tokenizer(texts, return_tensors="pt", padding="max_length", truncation=True, max_length=256)###
        #batch_image = self.processor(images=images, return_tensors="pt")
        #batch = {**batch_text, **batch_image}


        labels = batch["input_ids"].clone()
        if self.processor.tokenizer.pad_token_id is not None:
            labels[labels == self.processor.tokenizer.pad_token_id] = -100
        batch["labels"] = labels

        return batch

data_collator = LLavaDataCollator(processor)



training_args = TrainingArguments(
    output_dir="llava-1.5-7b-hf-ft-mix-vsft",
    report_to="tensorboard",
    learning_rate=1e-5,  #1.4e-5,
    #lr_scheduler_type="constant",
    per_device_train_batch_size=3, #4
    gradient_accumulation_steps=1, #1
    #gradient_clip_val= 1.0,  ####
    warmup_steps=30,  ###
    logging_steps=10,
    num_train_epochs=1, #2
    #push_to_hub=True,
    evaluation_strategy="no",  # Disable evaluation
    gradient_checkpointing=True,
    remove_unused_columns=False,
    fp16=True, #True
    bf16=False,  #False
    save_steps=200,  # Save checkpoint every 200 steps (adjust based on your training duration)
    save_total_limit=3,  # Keep only the last 3 checkpoints
)
# training_args = TrainingArguments(
#     output_dir="llava-1.5-7b-hf-ft-mix-vsft",
#     report_to="tensorboard",
#     learning_rate=1e-5,  #1e-4 at 1st epoch, then 1e-5 on 2nd epoch
#     #lr_scheduler_type="constant",
#     per_device_train_batch_size=4,
#     gradient_accumulation_steps=1, #1
#     warmup_steps=30,
#     logging_steps=10,
#     num_train_epochs=3, #2
#     #push_to_hub=True,
#     evaluation_strategy="no",  # Disable evaluation
#     gradient_checkpointing=True,
#     remove_unused_columns=False,
#     fp16=True, #True
#     bf16=False,  #False
#     save_steps=200,  # Save checkpoint every 500 steps (adjust based on your training duration)
#     save_total_limit=3,  # Keep only the last 3 checkpoints
# )



# lora_config = LoraConfig(
#     r=64,
#     lora_alpha=16,
#     target_modules=["q_proj", "v_proj"]
# )
def find_all_linear_names(model):  ###
    cls = torch.nn.Linear
    lora_module_names = set()
    multimodal_keywords = ['multi_modal_projector', 'vision_model']
    for name, module in model.named_modules():
        if any(mm_keyword in name for mm_keyword in multimodal_keywords):
            continue
        if isinstance(module, cls):
            names = name.split('.')
            lora_module_names.add(names[0] if len(names) == 1 else names[-1])

    if 'lm_head' in lora_module_names: # needed for 16-bit
        lora_module_names.remove('lm_head')
    return list(lora_module_names)
print(find_all_linear_names(model))
lora_config = LoraConfig( ###
    r=64,  #64
    lora_alpha=64,  #16
    lora_dropout=0.1,
    target_modules=find_all_linear_names(model),
    init_lora_weights='gaussian',
    #task_type="CAUSAL_LM"
)
model= prepare_model_for_kbit_training(model) ###
model= get_peft_model(model, lora_config) ###


trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    #eval_dataset=val_dataset,
    peft_config=lora_config,
    dataset_text_field="text",  # need a dummy field
    tokenizer=tokenizer,
    data_collator=data_collator,
    dataset_kwargs={"skip_prepare_dataset": True},
)

trainer.train()
#trainer.train(resume_from_checkpoint=True)

`low_cpu_mem_usage` was None, now default to True since model is quantized.


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Some kwargs in processor config are unused and will not have any effect: num_additional_image_tokens. 
C:\Users\User\.conda\envs\cuda\lib\site-packages\transformers\training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


['k_proj', 'gate_proj', 'up_proj', 'v_proj', 'down_proj', 'o_proj', 'q_proj']


C:\Users\User\.conda\envs\cuda\lib\site-packages\huggingface_hub\utils\_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': dataset_text_field, dataset_kwargs. Will not be supported from version '1.0.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
C:\Users\User\.conda\envs\cuda\lib\site-packages\trl\trainer\sft_trainer.py:292: UserWarning: You didn't pass a `max_seq_length` argument to the SFTTrainer, this will default to 1024
  warnings.warn(
C:\Users\User\.conda\envs\cuda\lib\site-packages\trl\trainer\sft_trainer.py:321: UserWarning: You passed a `dataset_text_field` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
C:\Users\User\.conda\envs\cuda\lib\site-packages\trl\trainer\sft_trainer.py:327: UserWarning: You passed a `dataset_kwargs` argument to the SFTTrainer, the value you passed will over

Step,Training Loss
10,14.860200
20,13.741500
30,11.482500
40,8.518600
50,5.928500
60,4.737200
70,4.450200
80,4.312600
90,4.235000
100,4.177600


C:\Users\User\.conda\envs\cuda\lib\site-packages\torch\utils\checkpoint.py:92: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
C:\Users\User\.conda\envs\cuda\lib\site-packages\torch\utils\checkpoint.py:295: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with torch.enable_grad(), device_autocast_ctx, torch.cpu.amp.autocast(**ctx.cpu_autocast_kwargs):  # type: ignore[attr-defined]
C:\Users\User\.conda\envs\cuda\lib\site-packages\torch\utils\checkpoint.py:92: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  warnings.warn(
C:\Users\User\.conda\envs\cuda\lib\site-packages\torch\utils\checkpoint.py:295: FutureWarning: `torch.cpu.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cpu', args...)` instead.
  with torch.enable_grad(), device_autocast_ctx, torch.cpu.amp.autocast(**ctx.cpu_autocast_kwargs):  # type: ignore[

TrainOutput(global_step=1697, training_loss=3.9681380456801927, metrics={'train_runtime': 12074.4571, 'train_samples_per_second': 0.422, 'train_steps_per_second': 0.141, 'total_flos': 1.3201346750360371e+17, 'train_loss': 3.9681380456801927, 'epoch': 1.0})

In [ ]:
model

PeftModel(
  (base_model): LoraModel(
    (model): LlavaForConditionalGeneration(
      (vision_tower): CLIPVisionModel(
        (vision_model): CLIPVisionTransformer(
          (embeddings): CLIPVisionEmbeddings(
            (patch_embedding): Conv2d(3, 1024, kernel_size=(14, 14), stride=(14, 14), bias=False)
            (position_embedding): Embedding(577, 1024)
          )
          (pre_layrnorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (encoder): CLIPEncoder(
            (layers): ModuleList(
              (0-23): 24 x CLIPEncoderLayer(
                (self_attn): CLIPSdpaAttention(
                  (k_proj): Linear4bit(in_features=1024, out_features=1024, bias=True)
                  (v_proj): lora.Linear4bit(
                    (base_layer): Linear4bit(in_features=1024, out_features=1024, bias=True)
                    (lora_dropout): ModuleDict(
                      (default): Identity()
                    )
                    (lora_A): ModuleDict

In [ ]:
# At the end of training, save the processor as well
output_dir = r"C:\Users\User\Desktop\jupiter\Thesis\LLAVA finetuning\llava-1.5-7b-hf-ft-mix-vsft"
processor.save_pretrained(output_dir)  # Save processor along with the model

# Save model and tokenizer as well
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

('C:\\Users\\User\\Desktop\\jupiter\\Thesis\\LLAVA finetuning\\llava-1.5-7b-hf-ft-mix-vsft\\tokenizer_config.json',
 'C:\\Users\\User\\Desktop\\jupiter\\Thesis\\LLAVA finetuning\\llava-1.5-7b-hf-ft-mix-vsft\\special_tokens_map.json',
 'C:\\Users\\User\\Desktop\\jupiter\\Thesis\\LLAVA finetuning\\llava-1.5-7b-hf-ft-mix-vsft\\tokenizer.json')

In [ ]:
from transformers import LlavaForConditionalGeneration, AutoTokenizer
#processor = AutoProcessor.from_pretrained("llava-hf/llava-1.5-7b-hf")  # Try loading from a base model
# Define the path where the model was saved (output directory from training)
output_dir = r"C:\Users\User\Desktop\jupiter\Thesis\LLAVA finetuning\llava-1.5-7b-hf-ft-mix-vsft"  # This is the folder where the model was saved

# Load the trained model
#model = LlavaForConditionalGeneration.from_pretrained(output_dir)

LLAVA_CHAT_TEMPLATE = """A chat between a curious user and an artificial intelligence assistant. The assistant gives helpful, detailed, and polite answers to the user's questions. {% for message in messages %}{% if message['role'] == 'user' %}USER: {% else %}ASSISTANT: {% endif %}{% for item in message['content'] %}{% if item['type'] == 'text' %}{{ item['text'] }}{% elif item['type'] == 'image' %}<image>{% endif %}{% endfor %}{% if message['role'] == 'user' %} {% else %}{{eos_token}}{% endif %}{% endfor %}"""
# quantization_config = BitsAndBytesConfig(
#     load_in_4bit=True,
# )
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16
)
model = LlavaForConditionalGeneration.from_pretrained(output_dir,
                                                      quantization_config=quantization_config,
                                                      torch_dtype=torch.float16)
tokenizer = AutoTokenizer.from_pretrained(output_dir)
tokenizer.chat_template = LLAVA_CHAT_TEMPLATE
processor = AutoProcessor.from_pretrained(output_dir)
processor.tokenizer = tokenizer

# Load the tokenizer
#tokenizer = AutoTokenizer.from_pretrained(output_dir)

# Optionally, load the processor (if you were using one)
#from transformers import AutoProcessor
#processor = AutoProcessor.from_pretrained(output_dir)

`low_cpu_mem_usage` was None, now default to True since model is quantized.


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [ ]:
from PIL import Image
import requests
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

#url = "https://github.com/haotian-liu/LLaVA/blob/1a91fc274d7c35a9b50b3cb29c4247ae5837ce39/images/llava_logo.png?raw=true"
#url = "https://4.img-dpreview.com/files/p/E~TS590x0~articles/3925134721/0266554465.jpeg"
#url = "https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcQjoGc8msuwHUOC5ikpOXy6SFMtGu_ODkttuw&s"
#url = "https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcSSvI5iKl0h2yyJJ2kIwVqoBbsFkMUPXwMuwA&s"
url = "https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcSOxAXZ_9gHR5JvmXJEArtCxso5FMlE2WGURw&s"
image = Image.open(requests.get(url, stream=True).raw)
#prompt = "[INST] <image>\nWhat is shown in this image? [/INST]"
#prompt = "<image>\nWhat preventive steps can be taken?"
prompt = "[INST] <image>\nExplain the image? [/INST]"

inputs = processor(prompt, image, return_tensors="pt").to(device)
output = model.generate(**inputs, max_new_tokens=50)
print(processor.decode(output[0], skip_special_tokens=True))

[INST]  
Explain the image? [/INST]


In [ ]:
from PIL import Image
# Load image from local path (replace with your local image path)
image_path = r"C:\Users\User\Pictures\Thesis\VQA dataset added\combined-img\ISIC_0025452.jpg"  # Replace with the local image path
image = Image.open(image_path)  # Open the image from the local path
# Prompt text (modify as needed)
#prompt = "<image>\nWhat precautions can reduce its occurrence?"
prompt = "<image>\nWhat makes this lesion identifiable as vascular?"

# Ensure the device (GPU or CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# Tokenize the prompt and process the image
inputs = processor(prompt, image, return_tensors="pt").to(device)
# Generate output from the model
output = model.generate(**inputs, max_new_tokens=50)

# Decode and print the output
print(processor.decode(output[0], skip_special_tokens=True))


What makes this lesion identifiable as vascular? 


In [ ]:
############################# Alt inference

In [ ]:
#MODEL_ID = "llava-hf/llava-v1.6-mistral-7b-hf"
MODEL_ID = "llava-hf/llava-1.5-7b-hf"
REPO_ID = r"C:\Users\User\Desktop\jupiter\Thesis\LLAVA finetuning\llava-1.5-7b-hf-ft-mix-vsft"

processor = AutoProcessor.from_pretrained(MODEL_ID)

# Define quantization config
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16
)
# Load the base model with adapters on top
model = LlavaForConditionalGeneration.from_pretrained(
    REPO_ID,
    torch_dtype=torch.float16,
    quantization_config=quantization_config,
)

Some kwargs in processor config are unused and will not have any effect: num_additional_image_tokens. 
`low_cpu_mem_usage` was None, now default to True since model is quantized.


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [ ]:
from PIL import Image
# Load image from local path (replace with your local image path)
image_path = r"C:\Users\User\Pictures\Thesis\VQA dataset added\combined-img\10_VI-chickenpox (27).jpg"  # Replace with the local image path
image = Image.open(image_path)  # Open the image from the local path

prompt = f"[INST] <image>\nHow serious is this disease? [/INST]"
max_output_token = 30
inputs = processor(prompt, image, return_tensors="pt").to("cuda:0")
output = model.generate(**inputs, max_new_tokens=max_output_token)
response = processor.decode(output[0], skip_special_tokens=True)
print(response)

[INST]  
How serious is this disease? [/INST]


In [ ]:
############################ Fixed inference

In [ ]:
import torch
from PIL import Image
from transformers import LlavaForConditionalGeneration, AutoTokenizer, AutoProcessor, BitsAndBytesConfig

# === 1. Setup ===
output_dir = r"C:\Users\User\Desktop\jupiter\Thesis\LLAVA finetuning\llava-1.5-7b-hf-ft-mix-vsft"

# === 2. Load model, tokenizer, and processor ===
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

model = LlavaForConditionalGeneration.from_pretrained(output_dir,
                                                      quantization_config=quant_config,
                                                      torch_dtype=torch.float16)
tokenizer = AutoTokenizer.from_pretrained(output_dir)
processor = AutoProcessor.from_pretrained(output_dir)

# === 3. Apply chat template to tokenizer ===
#LLAVA_CHAT_TEMPLATE = """A chat between a curious user and an artificial intelligence assistant. The assistant gives helpful, detailed, and polite answers to the user's questions. {% for message in messages %}{% if message['role'] == 'user' %}USER: {% else %}ASSISTANT: {% endif %}{% for item in message['content'] %}{% if item['type'] == 'text' %}{{ item['text'] }}{% elif item['type'] == 'image' %}<image>{% endif %}{% endfor %}{% if message['role'] == 'user' %} {% else %}{{eos_token}}{% endif %}{% endfor %}"""
LLAVA_CHAT_TEMPLATE = """{% for message in messages %}{% if message['role'] == 'user' %}USER: {% else %}ASSISTANT: {% endif %}{% for item in message['content'] %}{% if item['type'] == 'text' %}{{ item['text'] }}{% elif item['type'] == 'image' %}<image>{% endif %}{% endfor %}{% if message['role'] == 'user' %} {% else %}{{eos_token}}{% endif %}{% endfor %}"""
tokenizer.chat_template = LLAVA_CHAT_TEMPLATE
processor.tokenizer = tokenizer


`low_cpu_mem_usage` was None, now default to True since model is quantized.


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [ ]:
# === 4. Load image ===
image_path = r"C:\Users\User\Pictures\Thesis\VQA dataset added\combined-img\83_VI-chickenpox (4).jpg"
##ISIC_0025452.jpg
##8_VI-chickenpox (1).jpg
##ISIC_7614036.jpg
image = Image.open(image_path)

# === 5. Prepare message in chat format ===
messages = [
    {
        "role": "user",
        "content": [
            {"type": "text", "text": "is it infectious"},
            {"type": "image"}
        ]
    }
]

# === 6. Generate chat-formatted prompt ===
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

# === 7. Prepare inputs ===
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
#model = model.to(device)

inputs = processor(image, prompt, return_tensors="pt").to(device)

# === 8. Generate output ===
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=1000)

# === 9. Decode and print output ===
response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print("\n🧠 Model Response:\n", response)



🧠 Model Response:
 USER: is it infectious   ASSISTANT: Viral
